# Exploración para el módulo de *feature engineering*

Este cuaderno responde al punto 11 de la Parte B. Parte exclusivamente de los Parquet y manifiesto producidos por la ingesta reproducible del punto 10; no lee CSV crudos ni ejecuta el experimento. Su propósito es decidir variables reutilizables, comprobar su cobertura y documentar por qué son causales. Solo calcula agregados por estación-hora y no muestra viajes ni atributos personales.

Antes de ejecutarlo, desde la raíz del repositorio se requiere `uv sync --frozen --extra notebooks` y haber ejecutado `uv run --frozen python -m proyecto_mibici.dataset procesar-datos`.

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if not (ROOT / 'proyecto_mibici').is_dir():
    raise RuntimeError('Abra el cuaderno desde la raíz del repositorio o notebooks/.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
from proyecto_mibici.config import ProjectConfig
from proyecto_mibici.features import (
    RepositorioParquet,
    construir_perfil_semanal,
    construir_variables_horarias,
)

config = ProjectConfig(root=ROOT)
repository = RepositorioParquet(config)
manifest = repository.verify_base()
pd.Series({
    'data_version': manifest['data_version'],
    'viajes_limpios': manifest['resumen']['n_viajes_limpios'],
    'rango': f"{manifest['rango_fechas']['inicio']} a {manifest['rango_fechas']['fin']}",
})

2026-10-22 21:48:02.869 | INFO     | proyecto_mibici.config:<module>:14 - PROJ_ROOT path is: /Users/beibibumer/Documents/GitHub/TC5061-Proyecto-Equipo25


data_version           09a6a584dd5d
viajes_limpios             13621127
rango             2023-10 a 2026-09
dtype: object

## 1. Unidad temporal y cobertura causal

La unidad común es estación-hora. La función del repositorio completa el calendario solo después de verificar que los meses fuente están completos. Sin embargo, no transforma cualquier ausencia en actividad: una hora es `elegible` únicamente si en las 168 horas **anteriores** hubo actividad observada. Por ello, los ceros dentro de cobertura son analíticos; fuera de cobertura son desconocidos. Esta regla es necesaria para P1 y P2 y se conserva para construir P3.

In [2]:
coverage = []
for station, hours in repository.station_hours():
    coverage.append({
        'estacion': station,
        'horas_calendario': len(hours),
        'horas_elegibles': int(hours.elegible.sum()),
        'proporcion_elegible': float(hours.elegible.mean()),
        'salidas': int(hours.salidas.sum()),
        'llegadas': int(hours.llegadas.sum()),
    })
coverage = pd.DataFrame(coverage)
coverage[['horas_calendario', 'horas_elegibles', 'salidas', 'llegadas']].sum(), coverage.proporcion_elegible.describe()

(horas_calendario    12441792
 horas_elegibles      9687875
 salidas             13621127
 llegadas            13621111
 dtype: int64,
 count    473.000000
 mean       0.778656
 std        0.372703
 min        0.006425
 25%        0.906440
 50%        0.993613
 75%        0.993613
 max        0.993613
 Name: proporcion_elegible, dtype: float64)

**Resultado observado el 22 de octubre de 2026.** La base `09a6a584dd5d` contiene 473 estaciones observadas y 9,687,875 estación-horas elegibles. La mediana de cobertura elegible por estación fue aproximadamente 99.36 %, aunque hay estaciones con cobertura reducida. Por eso se conserva la máscara por estación-hora y no se imputa la historia faltante.

## 2. Variables para P1 y P2

Para **P1** (salidas de la siguiente hora) y **P2** (clase de presión de la siguiente hora), el calendario está disponible antes de predecir y describe patrones repetitivos: hora, día de semana, fin de semana y mes. Se codifican hora y día con seno/coseno para que 23:00 esté cerca de 00:00 y domingo cerca de lunes.

Los flujos históricos se limitan a rezagos de 1, 3, 24 y 168 horas, medias de 3, 24 y 168 horas y balances acumulados de 3 y 24 horas. Los rezagos de 24/168 comprueban recurrencia diaria/semanal; los cortos y las medias capturan actividad reciente. El balance `salidas - llegadas` es pertinente solo para el **proxy** de P2: expresa dirección de flujo, nunca inventario real. Todas estas variables usan `shift`, por lo que la fila de la hora $t$ no lee los viajes ocurridos en $t$ ni después.

In [3]:
feature_audit = []
for station, hours in repository.station_hours():
    variables = construir_variables_horarias(hours)
    weekly_reference = hours.elegible & hours.elegible.shift(config.weekly_lag, fill_value=False)
    complete = variables.notna().all(axis=1) & hours.elegible
    feature_audit.append({
        'estacion': station,
        'referencia_semanal_p1': int(weekly_reference.sum()),
        'features_completas_p1_p2': int(complete.sum()),
    })
feature_audit = pd.DataFrame(feature_audit)
feature_audit.drop(columns='estacion').sum(), feature_audit.features_completas_p1_p2.describe()

(referencia_semanal_p1       9604382
 features_completas_p1_p2    9603707
 dtype: int64,
 count      473.000000
 mean     20303.820296
 std       9799.809582
 min          1.000000
 25%      23675.000000
 50%      25968.000000
 75%      25968.000000
 max      25968.000000
 Name: features_completas_p1_p2, dtype: float64)

**Resultado observado.** Hay 9,604,382 estación-horas con referencia de la misma hora semanal para P1 y 9,603,707 con el conjunto completo de variables históricas de P1/P2. La diferencia se conserva como exclusión por historial, no como imputación. Las funciones implementadas son `construir_features_calendario`, `construir_features_flujo_historico` y `construir_variables_horarias`.

## 3. Perfil semanal para P3

P3 no pronostica una hora futura: describe estaciones dentro de cada partición temporal. Para comparar la **forma** de uso y no el volumen, se calculan 336 tasas: 7 días × 24 horas × 2 direcciones. Cada conteo se divide entre las horas elegibles de su franja; después, el vector conjunto se normaliza a uno. Una estación se excluye en una partición si alguna de las 168 franjas no tiene exposición o si no tiene eventos. Así una ausencia de observación no se confunde con uso cero y no se usan coordenadas ni capacidad como variables de agrupamiento.

In [4]:
profile_audit = []
for station, hours in repository.station_hours():
    for split, (start, end) in config.splits.items():
        profile, audit = construir_perfil_semanal(hours.loc[(hours.index >= start) & (hours.index < end)])
        profile_audit.append({'estacion': station, 'conjunto': split, **audit})
profile_audit = pd.DataFrame(profile_audit)
profile_audit.groupby(['conjunto', 'estado']).size().unstack(fill_value=0)

estado,cobertura_semanal_incompleta,incluida
conjunto,,
test,2,471
train,102,371
validation,106,367


**Resultado observado.** Se obtuvieron perfiles completos para 371 estaciones en entrenamiento, 367 en validación y 471 en prueba. Esas diferencias justifican auditar y reportar exclusiones por partición, en lugar de asignar perfiles a estaciones con franjas desconocidas. Las funciones implementadas son `construir_perfil_semanal` y `proporcion_fin_semana`; esta última resume sábado-domingo para el baseline descriptivo de P3.

## Decisión trasladada a `features.py`

El módulo concentra la agregación estación-hora, la regla de elegibilidad, variables de calendario, resúmenes históricos y perfiles semanales. Los modelos reciben esos resultados, pero no definen transformaciones. Para P1/P2 se construyen variables antes de alinear el objetivo de una hora posterior; para P3, cada partición se perfila por separado. La validación y prueba nunca participan en el ajuste de umbrales, transformaciones aprendidas o grupos.